# Week 11: Choosing Representations and Building Responsible Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/11/Week_11_Choosing_Representations_Responsible_Systems.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)

## Final-session goals

By the end of this session you should be able to:

- choose an architecture from the structure of a problem, not from fashion;
- explain what representation an MLP, CNN, RNN, or transformer is biased to learn;
- compare models using performance, representation behavior, compute, and failure evidence;
- audit confident errors, class slices, and distribution shift;
- sketch a complete ML system from data to monitoring;
- write a compact model/system card with intended use, limitations, and risks.

The course closes with one decision loop:

`problem -> data structure -> representation -> architecture -> evidence -> system -> monitoring`

## Architecture decision map

| Data/problem structure | Useful starting point | Representation bias | Common trap |
|---|---|---|---|
| Fixed-size measurements or engineered features | MLP | nonlinear combinations of input dimensions | ignoring feature scale or data leakage |
| Images or spatial grids | CNN | local patterns, translation-aware hierarchies | relying on background shortcuts |
| Ordered streams with compact recurrent memory | RNN/LSTM | evolving hidden state | losing long-range information |
| Tokens requiring broad contextual interaction | Transformer | context-dependent token representations | assuming attention is explanation |
| Small labeled dataset with related pretrained model | Frozen features / fine-tuning / PEFT | reuse and adapt an existing representation | fine-tuning everything without evidence |

These are starting points, not laws. Compute, data volume, latency, interpretability, and risk can change the decision.

### External anchors

| Topic | Resource | Why it helps |
|---|---|---|
| Parameter-efficient adaptation | [Hugging Face PEFT quicktour](https://huggingface.co/docs/peft/quicktour) | Shows how adapters can update a small fraction of a pretrained model. |
| LoRA | [Hugging Face LoRA reference](https://huggingface.co/docs/peft/package_reference/lora) | Concrete description of low-rank adaptation. |
| Model documentation | [Hugging Face Model Cards](https://huggingface.co/docs/hub/model-cards) | Practical structure for intended use, evaluation, limitations, and sharing. |
| Responsible AI systems | [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework) | Connects model work to governance, measurement, and risk management. |
| API deployment | [FastAPI deployment concepts](https://fastapi.tiangolo.com/deployment/concepts/) | Shows that serving a model is only one part of operating it. |

---

## Environment and final technical case study

We use the built-in 8x8 digits dataset. Two models solve the same task:

- an MLP sees a flat vector;
- a CNN sees a spatial grid.

This lets us compare architecture bias, learned representations, errors, and robustness without downloads or long training.

In [ ]:
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def set_seed(seed=11):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(11)

In [ ]:
digits = load_digits()
X_images = (digits.images.astype("float32") / 16.0)
y = digits.target.astype("int64")

X_train_full_img, X_test_img, y_train_full, y_test = train_test_split(
    X_images, y, test_size=0.25, random_state=11, stratify=y
)
X_train_img, X_val_img, y_train, y_val = train_test_split(
    X_train_full_img, y_train_full, test_size=0.20, random_state=11, stratify=y_train_full
)

# Both models receive the same 0-1 pixel values; only the shape/architecture differs.
X_train_flat = X_train_img.reshape(len(X_train_img), -1).astype("float32")
X_val_flat = X_val_img.reshape(len(X_val_img), -1).astype("float32")
X_test_flat = X_test_img.reshape(len(X_test_img), -1).astype("float32")

train = {
    "flat": torch.tensor(X_train_flat),
    "image": torch.tensor(X_train_img[:, None, :, :]),
    "y": torch.tensor(y_train),
}
val = {
    "flat": torch.tensor(X_val_flat),
    "image": torch.tensor(X_val_img[:, None, :, :]),
    "y": torch.tensor(y_val),
}
test = {
    "flat": torch.tensor(X_test_flat),
    "image": torch.tensor(X_test_img[:, None, :, :]),
    "y": torch.tensor(y_test),
}

print("train/validation/test:", len(y_train), len(y_val), len(y_test))
fig, axes = plt.subplots(2, 8, figsize=(9, 2.7))
for ax, image, label in zip(axes.ravel(), X_train_img[:16], y_train[:16]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("Final case study: handwritten digits")
plt.tight_layout()
plt.show()

---

## 1. Same task, different representation bias

The MLP receives 64 unrelated positions unless training discovers relationships. The CNN receives an 8x8 grid and explicitly reuses local filters across space.

In [ ]:
class DigitMLP(nn.Module):
    def __init__(self, rep_dim=24):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, rep_dim), nn.ReLU(),
        )
        self.classifier = nn.Linear(rep_dim, 10)

    def forward(self, x, return_rep=False):
        rep = self.features(x)
        logits = self.classifier(rep)
        return (logits, rep) if return_rep else logits


class DigitCNN(nn.Module):
    def __init__(self, rep_dim=24):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.project = nn.Sequential(nn.Flatten(), nn.Linear(16 * 2 * 2, rep_dim), nn.ReLU())
        self.classifier = nn.Linear(rep_dim, 10)

    def forward(self, x, return_rep=False):
        rep = self.project(self.conv(x))
        logits = self.classifier(rep)
        return (logits, rep) if return_rep else logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def batches(X, y, batch_size=128, seed=0, shuffle=True):
    idx = np.arange(len(y))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for start in range(0, len(idx), batch_size):
        take = idx[start:start + batch_size]
        yield X[take].to(device), y[take].to(device)


@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    all_probs, all_reps = [], []
    total_loss = 0.0
    for xb, yb in batches(X, y, shuffle=False):
        logits, rep = model(xb, return_rep=True)
        total_loss += F.cross_entropy(logits, yb, reduction="sum").item()
        all_probs.append(F.softmax(logits, dim=1).cpu())
        all_reps.append(rep.cpu())
    probs = torch.cat(all_probs)
    pred = probs.argmax(dim=1)
    return {
        "loss": total_loss / len(y),
        "accuracy": float((pred == y).float().mean()),
        "probs": probs,
        "pred": pred,
        "representations": torch.cat(all_reps),
    }


def train_model(model_factory, Xtr, ytr, Xval, yval, epochs=25, lr=0.01, seed=11):
    set_seed(seed)
    model = model_factory().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_acc": [], "val_acc": []}
    start = time.time()
    for epoch in range(epochs):
        model.train()
        for xb, yb in batches(Xtr, ytr, seed=seed + epoch):
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            opt.step()
        history["train_acc"].append(evaluate(model, Xtr, ytr)["accuracy"])
        history["val_acc"].append(evaluate(model, Xval, yval)["accuracy"])
    return model, history, time.time() - start

In [ ]:
mlp, mlp_history, mlp_seconds = train_model(
    DigitMLP, train["flat"], train["y"], val["flat"], val["y"], seed=11
)
cnn, cnn_history, cnn_seconds = train_model(
    DigitCNN, train["image"], train["y"], val["image"], val["y"], seed=11
)

mlp_val = evaluate(mlp, val["flat"], val["y"])
cnn_val = evaluate(cnn, val["image"], val["y"])
mlp_eval = evaluate(mlp, test["flat"], test["y"])
cnn_eval = evaluate(cnn, test["image"], test["y"])

comparison = [
    {"model": "MLP", "val_accuracy": mlp_val["accuracy"], "final_test_accuracy": mlp_eval["accuracy"], "parameters": count_parameters(mlp), "seconds": mlp_seconds},
    {"model": "CNN", "val_accuracy": cnn_val["accuracy"], "final_test_accuracy": cnn_eval["accuracy"], "parameters": count_parameters(cnn), "seconds": cnn_seconds},
]
for row in comparison:
    print(row)

plt.figure(figsize=(7, 4))
plt.plot(mlp_history["val_acc"], label="MLP")
plt.plot(cnn_history["val_acc"], label="CNN")
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Same task, different architecture bias")
plt.legend()
plt.tight_layout()
plt.show()

### Interpretation pause

- Which architecture would validation evidence select?
- Did the architecture with the stronger spatial bias use more or fewer parameters?
- Did the final test result agree with the validation decision?
- Is a small accuracy difference enough to decide?
- What deployment constraints could reverse the choice?

**Evaluation discipline:** use validation evidence to choose. Treat the test score as a final audit, not a knob to optimize by repeatedly editing and re-running.

---

## 2. Compare learned representation geometry

Accuracy tells us whether the classifier works. Representation geometry helps us inspect what became easy for the final linear head.

In [ ]:
def plot_representation_pca(evaluations, labels):
    fig, axes = plt.subplots(1, len(evaluations), figsize=(12, 5), constrained_layout=True)
    for ax, (name, result) in zip(axes, evaluations.items()):
        coords = PCA(n_components=2, random_state=0).fit_transform(result["representations"].numpy())
        scatter = ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", s=14, alpha=0.75)
        ax.set_title(f"{name} representation")
        ax.set_xlabel("PCA 1")
        ax.set_ylabel("PCA 2")
    fig.colorbar(scatter, ax=axes, fraction=0.025, label="digit")
    plt.show()

plot_representation_pca({"MLP": mlp_eval, "CNN": cnn_eval}, y_test)

---

## 3. Coding block 1: architecture decision lab

Choose one controlled change:

- reduce the representation dimension;
- widen the MLP;
- add another CNN channel;
- increase or reduce training epochs.

Then compare four kinds of evidence:

1. validation accuracy;
2. parameter count;
3. training time;
4. representation PCA.

Keep the test split out of this tuning loop. Use it only after the model choice is fixed.

**Decision prompt:** Which model would you choose for a mobile app, a server API, and a high-stakes review workflow? The answer may differ.

In [ ]:
# TODO: modify one model or training setting.
student_model, student_history, student_seconds = train_model(
    lambda: DigitCNN(rep_dim=12),
    train["image"], train["y"],
    val["image"], val["y"],
    epochs=22,
    lr=0.01,
    seed=21,
)
student_val = evaluate(student_model, val["image"], val["y"])

print({
    "model": "student candidate",
    "validation_accuracy": round(student_val["accuracy"], 4),
    "parameters": count_parameters(student_model),
    "seconds": round(student_seconds, 2),
})
plot_representation_pca({"baseline CNN": cnn_val, "student candidate": student_val}, y_val)

---

## 4. Failure audit: aggregate scores hide structure

A responsible decision looks beyond overall accuracy:

- Which classes fail?
- Which mistakes are highly confident?
- Does performance change under plausible shift?
- Which users or subgroups bear the errors?

In [ ]:
def per_class_accuracy(y_true, pred):
    rows = []
    for cls in range(10):
        mask = y_true == cls
        rows.append((cls, float((pred[mask] == y_true[mask]).float().mean()), int(mask.sum())))
    return rows

for name, result in [("MLP", mlp_eval), ("CNN", cnn_eval)]:
    print("\n", name)
    for cls, acc, n in per_class_accuracy(test["y"], result["pred"]):
        print(f"digit {cls}: accuracy={acc:.3f} n={n}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, result) in zip(axes, [("MLP", mlp_eval), ("CNN", cnn_eval)]):
    cm = confusion_matrix(y_test, result["pred"].numpy())
    ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{name} confusion matrix")
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
plt.tight_layout()
plt.show()

In [ ]:
def show_confident_errors(images, y_true, result, title, n=12):
    confidence, pred = result["probs"].max(dim=1)
    wrong = pred != y_true
    candidates = torch.where(wrong)[0]
    ranked = candidates[torch.argsort(confidence[candidates], descending=True)][:n]
    fig, axes = plt.subplots(2, 6, figsize=(10, 4))
    for ax, idx in zip(axes.ravel(), ranked):
        ax.imshow(images[int(idx)], cmap="gray_r")
        ax.set_title(f"true {int(y_true[idx])}\npred {int(pred[idx])} ({confidence[idx]:.2f})")
        ax.axis("off")
    for ax in axes.ravel()[len(ranked):]:
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_confident_errors(X_test_img, test["y"], cnn_eval, "CNN: most confident mistakes")

---

## 5. Distribution shift audit

We simulate a plausible input change: noisier, lower-contrast images. This is not a complete robustness test, but it demonstrates why deployment data must be monitored.

In [ ]:
rng = np.random.default_rng(11)
X_shift_img = np.clip(X_test_img * 0.65 + rng.normal(0, 0.16, X_test_img.shape), 0, 1).astype("float32")
X_shift_flat = X_shift_img.reshape(len(X_shift_img), -1).astype("float32")
shift = {
    "flat": torch.tensor(X_shift_flat),
    "image": torch.tensor(X_shift_img[:, None, :, :]),
    "y": test["y"],
}

mlp_shift = evaluate(mlp, shift["flat"], shift["y"])
cnn_shift = evaluate(cnn, shift["image"], shift["y"])

for name, clean_result, shift_result in [
    ("MLP", mlp_eval, mlp_shift),
    ("CNN", cnn_eval, cnn_shift),
]:
    print(name, {
        "clean_accuracy": round(clean_result["accuracy"], 4),
        "shift_accuracy": round(shift_result["accuracy"], 4),
        "drop": round(clean_result["accuracy"] - shift_result["accuracy"], 4),
    })

fig, axes = plt.subplots(2, 8, figsize=(9, 3.2))
for j in range(8):
    axes[0, j].imshow(X_test_img[j], cmap="gray_r")
    axes[1, j].imshow(X_shift_img[j], cmap="gray_r")
    axes[0, j].axis("off"); axes[1, j].axis("off")
axes[0, 0].set_ylabel("clean")
axes[1, 0].set_ylabel("shifted")
plt.suptitle("Synthetic deployment shift")
plt.tight_layout()
plt.show()

---

## 6. Coding block 2: failure and risk audit

Choose at least two:

- inspect the weakest digit class;
- compare confidence on correct vs incorrect examples;
- increase or decrease shift severity;
- define a subgroup such as curved digits (`0, 3, 6, 8, 9`) vs others;
- propose an abstention rule for low-confidence predictions.

Then write an operational decision:

> What should happen when the model is uncertain or the input looks shifted?

**Important:** the curved-vs-other digit split below demonstrates how to calculate a slice metric. It is not evidence of demographic fairness. A real fairness audit requires relevant groups, domain-specific harms, appropriate data governance, and enough examples for reliable estimates.

In [ ]:
# TODO: edit the subgroup or confidence threshold.
curved_digits = torch.tensor([0, 3, 6, 8, 9])
subgroup_mask = torch.isin(test["y"], curved_digits)
confidence, prediction = cnn_eval["probs"].max(dim=1)

for label, mask in [("curved", subgroup_mask), ("other", ~subgroup_mask)]:
    acc = float((prediction[mask] == test["y"][mask]).float().mean())
    print(label, "n=", int(mask.sum()), "accuracy=", round(acc, 4), "mean confidence=", round(float(confidence[mask].mean()), 4))

threshold = 0.80
accepted = confidence >= threshold
accepted_accuracy = float((prediction[accepted] == test["y"][accepted]).float().mean())
print({
    "threshold": threshold,
    "coverage": round(float(accepted.float().mean()), 4),
    "accepted_accuracy": round(accepted_accuracy, 4),
    "sent_to_review": int((~accepted).sum()),
})

---

## 7. From model to system

A trained network is one component:

```text
problem definition
    -> data collection and documentation
    -> versioned preprocessing
    -> training and experiment evidence
    -> model artifact and model card
    -> API or batch integration
    -> logging and monitoring
    -> drift / failure review
    -> retraining or rollback
```

Questions before deployment:

- What is the intended use and what is explicitly out of scope?
- What inputs should be rejected?
- What metric matters operationally?
- Who reviews uncertain or harmful outputs?
- What gets logged without violating privacy?
- What triggers rollback or retraining?

### Course coverage and next steps

This course built the modeling and decision foundations. Some production topics were introduced conceptually rather than implemented end to end.

| Area | What we practiced | Natural next step |
|---|---|---|
| Versioning | Git-based code/material workflow and reproducible configurations | dataset/model versioning with DVC or artifact stores |
| Experiment management | tracked configs, metrics, comparisons, and tuning logic | persistent MLflow, W&B, or Trackio projects |
| Adaptation | frozen features and fine-tuning decisions | real PEFT/LoRA on a pretrained foundation model |
| Deployment | API/system lifecycle and operational questions | package a model with FastAPI and Docker |
| Monitoring | slices, confidence, shift, abstention, and rollback logic | production telemetry, alerts, drift review, and incident response |

Knowing this boundary is part of system thinking: a three-hour demo is not production readiness.

## Architecture decision scenarios

For each scenario, choose a starting representation, architecture, validation evidence, and deployment risk.

| Scenario | Structure to notice |
|---|---|
| Hospital readmission prediction from fixed clinical measurements | tabular variables, missingness, high-stakes calibration |
| Defect detection from factory camera images | local spatial patterns, changing lighting, class imbalance |
| Equipment failure from sensor streams | temporal order, drift, rare events |
| Contract clause classification | long context, domain language, privacy |
| Customer-support assistant | retrieval, generative failure, monitoring, human escalation |

There may be several defensible answers. The quality of the reasoning matters more than naming the newest model.

---

## 8. Model/system card

Complete this compact card for the digit system or one scenario above.

| Field | Your answer |
|---|---|
| Intended use | TODO |
| Out-of-scope use | TODO |
| Input and output | TODO |
| Chosen representation/architecture | TODO |
| Training and validation evidence | TODO |
| Important slices or subgroups | TODO |
| Known failure modes | TODO |
| Shift/robustness evidence | TODO |
| Human review or fallback | TODO |
| Monitoring signals | TODO |
| Retraining/rollback trigger | TODO |
| Privacy, fairness, or safety risks | TODO |

---

## 9. Current directions: where the same ideas continue

- **Foundation and multimodal models:** larger pretrained representations reused across tasks.
- **PEFT / LoRA:** adapt a small number of parameters instead of updating the whole model.
- **Retrieval and tool use:** combine learned representations with external information and actions.
- **Smaller domain models:** trade broad capability for cost, latency, privacy, or specialization.
- **Quantization and efficient inference:** reduce memory and compute after training.
- **Evaluation as a system:** test tasks, slices, shift, safety, latency, and human workflows together.

The tools change. The durable questions remain:

1. What representation is being learned or reused?
2. What evidence says it works?
3. Where does it fail?
4. What system surrounds it?

## Course close

You began with a linear threshold unit and ended with representation-learning systems.

The full course path:

`linear decision -> nonlinear representation -> gradients -> training behavior -> repair -> CNN -> RNN/attention -> transformer -> experiment system -> responsible architecture decision`

A good practitioner does not merely train a model. They can explain:

- why this representation;
- why this architecture;
- why this evidence;
- why this deployment decision;
- and what happens when the model is wrong.